---
title: "Sprint 2: Algorytmy ANN"
subtitle: "Ewaluacja metod wyszukiwania dla problemu Particle Tracking"
author: ""
date: today
format: 
  pdf:
    toc: true
    number-sections: true
    colorlinks: true
    fig-format: pdf
    fig-width: 8
    fig-height: 5
execute:
  echo: false
  warning: false
---

In [1]:
#| output: false
%config InlineBackend.figure_format = 'pdf'
%load_ext autoreload
%autoreload 2

import sys
import gc
from pathlib import Path
import numpy as np
import faiss

notebook_dir = Path().resolve()
project_root = notebook_dir.parent
src_path = str(project_root / "src")

if src_path not in sys.path:
    sys.path.append(src_path)

from hep_tracking.models import knn_scipy_ckdtree
from hep_tracking.utils import pad_features, evaluate_ann_model, measure_execution_time
from hep_tracking.plots import plot_pareto_frontier, plot_crossover, plot_ann_scaling, plot_exact_vs_ann
from hep_tracking.ann_models import FaissIVFFlat, FaissIVFPQ, HnswGraph

print("Zależności załadowane pomyślnie!")

Zależności załadowane pomyślnie!


In [2]:
#| output: false
data_dir = project_root / "data"
dataset_name = "dataset_hard_100k.npz"
candidates_name = "candidates_hard_100k.npz"

data_path = data_dir / dataset_name
candidates_path = data_dir / candidates_name

if not data_path.exists() or not candidates_path.exists():
    raise FileNotFoundError("Brak danych! Upewnij się, że wywołałeś `make candidates`.")

features = np.load(data_path)["X"]
n_queries = features.shape[0]

true_indices = np.load(candidates_path)["indices"]
k_neighbors = true_indices.shape[1]

print(f"Załadowano dane: {n_queries} punktów w przestrzeni {features.shape[1]}D")
print(f"Szukamy {k_neighbors} najbliższych sąsiadów.")

Załadowano dane: 99950 punktów w przestrzeni 5D
Szukamy 5 najbliższych sąsiadów.


# Wprowadzenie
Celem drugiego sprintu projektu jest ocena wydajności algorytmów przybliżonego wyszukiwania najbliższych sąsiadów (ANN) w kontekście rekonstrukcji torów cząstek. Ze względu na ograniczenia sprzętowe akceleratorów (GPU) narzucające długość wektora $m \in \{1, 2, 3, 4, 8, 12...\}$, 5-wymiarowe dane wejściowe zostały rozszerzone do 8 wymiarów przy pomocy bezstratnej techniki Zero-Padding.

# Poszukiwanie Granicy Pareto (Recall vs QPS)
W pierwszym eksperymencie przeprowadzono przeszukiwanie siatki hiperparametrów (Grid Search) dla trzech głównych algorytmów: **FAISS IVFFlat**, **FAISS IVFPQ** (z Product Quantization) oraz **HNSW**. 

Celem było wyznaczenie kompromisu między czułością modelu (Recall) a jego przepustowością (Queries Per Second).

In [3]:
print("=== Przygotowanie danych (Zero-Padding do 8D) ===")
features_5d = np.load(data_path)["X"]
features = pad_features(features_5d, target_dim=8)
print(f"Dane gotowe: {features.shape}")

USE_GPU = True

print("=== Generowanie GLOBALNEGO ground truth (ignorując granice zdarzeń) ===")
print("Uwaga: candidates_hard_100k.npz nie nadaje się tutaj — te sąsiedztwa są")
print("liczone OSOBNO dla każdego zdarzenia, a ten benchmark przeszukuje cały zbiór naraz.")

_, true_indices = knn_scipy_ckdtree(features, k=k_neighbors)
print(f"Gotowe: {true_indices.shape[0]} zapytań x {true_indices.shape[1]} sąsiadów (dokładne, globalne)")

results = {
    "IVFFlat": {"recall": [], "qps": [], "labels": []},
    "IVFPQ": {"recall": [], "qps": [], "labels": []},
    "HNSW": {"recall": [], "qps": [], "labels": []}
}

nlist = 100

print("\n=== START IVFFlat ===")
for nprobe in [1, 2, 5, 10, 20, 50]:
    print(f" -> Inicjalizacja IVFFlat (nprobe={nprobe})...")
    model = FaissIVFFlat(nlist=nlist, nprobe=nprobe, use_gpu=USE_GPU)
    qps, recall = evaluate_ann_model(f"IVFFlat (nprobe={nprobe})", model, features, true_indices, k_neighbors)
    results["IVFFlat"]["qps"].append(qps)
    results["IVFFlat"]["recall"].append(recall)
    results["IVFFlat"]["labels"].append(f"np={nprobe}")
    del model
    gc.collect()

print("\n=== START IVFPQ (Teraz na GPU z m=8!) ===")
for nprobe in [1, 2, 5, 10, 20, 50, 100]:
    print(f" -> Inicjalizacja IVFPQ (nprobe={nprobe})...")
    model = FaissIVFPQ(nlist=nlist, m=8, nbits=8, nprobe=nprobe, use_gpu=USE_GPU)
    qps, recall = evaluate_ann_model(f"IVFPQ (nprobe={nprobe})", model, features, true_indices, k_neighbors)
    results["IVFPQ"]["qps"].append(qps)
    results["IVFPQ"]["recall"].append(recall)
    results["IVFPQ"]["labels"].append(f"np={nprobe}")
    del model
    gc.collect()

print("\n=== START HNSW ===")
m_graph = 16
for ef in [10, 20, 50, 100, 200]:
    print(f" -> Inicjalizacja HNSW (ef={ef})...")
    model = HnswGraph(m=m_graph, ef_construction=200, ef=ef)
    qps, recall = evaluate_ann_model(f"HNSW (ef={ef})", model, features, true_indices, k_neighbors)
    results["HNSW"]["qps"].append(qps)
    results["HNSW"]["recall"].append(recall)
    results["HNSW"]["labels"].append(f"ef={ef}")
    del model
    gc.collect()

print("\nCAŁY BENCHMARK ZOSTAŁ POMYŚLNIE ZAKOŃCZONY!")

=== Przygotowanie danych (Zero-Padding do 8D) ===
Dane gotowe: (99950, 8)
=== Generowanie GLOBALNEGO ground truth (ignorując granice zdarzeń) ===
Uwaga: candidates_hard_100k.npz nie nadaje się tutaj — te sąsiedztwa są
liczone OSOBNO dla każdego zdarzenia, a ten benchmark przeszukuje cały zbiór naraz.
Gotowe: 99950 zapytań x 5 sąsiadów (dokładne, globalne)

=== START IVFFlat ===
 -> Inicjalizacja IVFFlat (nprobe=1)...
Trenowanie i budowa indeksu: IVFFlat (nprobe=1)...
 -> QPS: 9,709,443 | Recall: 0.5015

 -> Inicjalizacja IVFFlat (nprobe=2)...
Trenowanie i budowa indeksu: IVFFlat (nprobe=2)...
 -> QPS: 7,861,971 | Recall: 0.5015

 -> Inicjalizacja IVFFlat (nprobe=5)...
Trenowanie i budowa indeksu: IVFFlat (nprobe=5)...
 -> QPS: 4,406,956 | Recall: 0.5015

 -> Inicjalizacja IVFFlat (nprobe=10)...
Trenowanie i budowa indeksu: IVFFlat (nprobe=10)...
 -> QPS: 2,377,124 | Recall: 0.5015

 -> Inicjalizacja IVFFlat (nprobe=20)...
Trenowanie i budowa indeksu: IVFFlat (nprobe=20)...
 -> QPS: 1,2

In [4]:
plot_pareto_frontier(
    results, 
    title="Wydajność algorytmów ANN: Recall vs. QPS (Przestrzeń 8D z Zero-Paddingiem)"
)

<Figure size 1000x700 with 1 Axes>

# Analiza Wąskiego Gardła: CPU vs GPU (Crossover N)
Karty graficzne oferują potężną moc zrównoleglonych obliczeń, jednak posiadają istotny narzut czasowy związany z transferem danych przez magistralę PCIe oraz kompilacją kerneli JIT. 

Poniższy eksperyment wyznacza punkt przecięcia (Crossover Point), w którym rozmiar zbioru danych $N$ staje się na tyle duży, że użycie GPU staje się uzasadnione czasowo.


In [5]:
target_sizes = {"1k": 1000, "10k": 10000, "100k": 100000, "1M": 1000000}
mode = "hard"
k_neighbors = 5
nprobe = 5

cpu_times = []
gpu_times = []
valid_sizes = []

print("=== WYSZUKIWANIE PUNKTU PRZECIĘCIA (CROSSOVER N): CPU vs GPU ===")

for size_label, n_points in target_sizes.items():
    filename = data_dir / f"dataset_{mode}_{size_label}.npz"

    if not filename.exists():
        print(f"[POMINIĘTO] Plik {filename.name} nie istnieje.")
        continue

    print(f"\nWczytywanie zbioru danych {size_label} ({n_points} punktów)...")
    features_5d = np.load(filename)["X"]
    features = pad_features(features_5d, target_dim=8)

    nlist = min(100, int(np.sqrt(features.shape[0])))

    model_cpu = FaissIVFFlat(nlist=nlist, nprobe=nprobe, use_gpu=False)
    model_cpu.build(features)
    
    best_cpu_time = measure_execution_time(lambda: model_cpu.query(features, k_neighbors), num_runs=3, warmup_runs=1)
    cpu_times.append(best_cpu_time)
    del model_cpu
    gc.collect()

    model_gpu = FaissIVFFlat(nlist=nlist, nprobe=nprobe, use_gpu=True)
    model_gpu.build(features)
    
    best_gpu_time = measure_execution_time(lambda: model_gpu.query(features, k_neighbors), num_runs=3, warmup_runs=1)
    gpu_times.append(best_gpu_time)
    del model_gpu
    gc.collect()

    valid_sizes.append(n_points)
    print(f" -> CPU Time: {best_cpu_time*1000:.2f} ms")
    print(f" -> GPU Time: {best_gpu_time*1000:.2f} ms")

plot_crossover(valid_sizes, cpu_times, gpu_times, title="Crossover N: CPU vs GPU")

=== WYSZUKIWANIE PUNKTU PRZECIĘCIA (CROSSOVER N): CPU vs GPU ===

Wczytywanie zbioru danych 1k (1000 punktów)...
 -> CPU Time: 3.31 ms
 -> GPU Time: 0.16 ms

Wczytywanie zbioru danych 10k (10000 punktów)...


WARNING clustering 995 points to 31 centroids: please provide at least 1209 training points
WARNING clustering 995 points to 31 centroids: please provide at least 1209 training points


 -> CPU Time: 33.54 ms
 -> GPU Time: 1.16 ms

Wczytywanie zbioru danych 100k (100000 punktów)...
 -> CPU Time: 343.99 ms
 -> GPU Time: 23.62 ms

Wczytywanie zbioru danych 1M (1000000 punktów)...
 -> CPU Time: 16458.48 ms
 -> GPU Time: 1852.37 ms


<Figure size 1000x600 with 1 Axes>

In [6]:
filename = data_dir / "dataset_hard_1k.npz"
features_5d = np.load(filename)["X"]
features_full = pad_features(features_5d, target_dim=8)

micro_sizes = [10, 100, 200, 500, 1000]
cpu_micro = []
gpu_micro = []

print("=== SPRAWDZENIE NIŻSZYCH WARTOŚCI N (< 1000) ===")

for size in micro_sizes:
    features = features_full[:size]
    nlist = min(100, int(np.sqrt(features.shape[0])))

    model_cpu = FaissIVFFlat(nlist=nlist, nprobe=5, use_gpu=False)
    model_cpu.build(features)
    best_cpu = measure_execution_time(lambda: model_cpu.query(features, k_neighbors), num_runs=5, warmup_runs=1)
    cpu_micro.append(best_cpu)
    del model_cpu

    model_gpu = FaissIVFFlat(nlist=nlist, nprobe=5, use_gpu=True)
    model_gpu.build(features)
    best_gpu = measure_execution_time(lambda: model_gpu.query(features, k_neighbors), num_runs=5, warmup_runs=1)
    gpu_micro.append(best_gpu)
    del model_gpu

    print(f"N={size}: CPU = {best_cpu*1000:.3f} ms | GPU = {best_gpu*1000:.3f} ms")

plot_crossover(micro_sizes, cpu_micro, gpu_micro, title="Crossover N: CPU vs GPU")

=== SPRAWDZENIE NIŻSZYCH WARTOŚCI N (< 1000) ===
N=10: CPU = 0.025 ms | GPU = 0.057 ms
N=100: CPU = 0.352 ms | GPU = 0.064 ms
N=200: CPU = 0.651 ms | GPU = 0.080 ms
N=500: CPU = 1.629 ms | GPU = 0.104 ms
N=1000: CPU = 3.252 ms | GPU = 0.157 ms


WARNING clustering 10 points to 3 centroids: please provide at least 117 training points
WARNING clustering 10 points to 3 centroids: please provide at least 117 training points
WARNING clustering 100 points to 10 centroids: please provide at least 390 training points
WARNING clustering 100 points to 10 centroids: please provide at least 390 training points
WARNING clustering 200 points to 14 centroids: please provide at least 546 training points
WARNING clustering 200 points to 14 centroids: please provide at least 546 training points
WARNING clustering 500 points to 22 centroids: please provide at least 858 training points
WARNING clustering 500 points to 22 centroids: please provide at least 858 training points
WARNING clustering 995 points to 31 centroids: please provide at least 1209 training points
WARNING clustering 995 points to 31 centroids: please provide at least 1209 training points


<Figure size 1000x600 with 1 Axes>

# Skalowalność Metod Przybliżonych
Wykorzystując optymalne parametry odczytane z granicy Pareto (gwarantujące czułość na poziomie ok. 95%), przeprowadzono badanie skalowalności czasowej algorytmów przybliżonych w funkcji rosnącego rozmiaru zbioru danych ($10^3 \le N \le 10^6$).

In [7]:
params = {
    "IVFFlat": {"nprobe": 20},
    "IVFPQ": {"nprobe": 20, "m": 8},
    "HNSW": {"ef": 20, "m_graph": 16}
}

run_algos = {
    "Exact_CPU": False,
    "Exact_GPU": False,
    "IVFFlat": True,
    "IVFPQ": True,
    "HNSW": True
}

results_time = {alg: [] for alg in run_algos.keys()}
valid_sizes = []

print(f"=== START EKSPERYMENTU: TIME vs N (USE_GPU dla FAISS ANN = {USE_GPU}) ===")

valid_datasets_count = sum(1 for size in target_sizes if (data_dir / f"dataset_{mode}_{size}.npz").exists())
active_algos_count = sum(run_algos.values())
total_tasks = valid_datasets_count * active_algos_count
current_task = 0

if total_tasks == 0:
    print("[BŁĄD] Brak zadań do wykonania (brak plików lub wszystkie algorytmy wyłączone).")
else:
    if run_algos["Exact_GPU"]:
        gpu_res = faiss.StandardGpuResources()

    for size_label, n_points in target_sizes.items():
        dataset_path = data_dir / f"dataset_{mode}_{size_label}.npz"
        if not dataset_path.exists():
            print(f"[POMIJAM] Nie znaleziono pliku {size_label}")
            continue
            
        print(f"\nŁadowanie zbioru danych {size_label}...")
        features_5d = np.load(dataset_path)["X"]
        
        features = pad_features(features_5d, target_dim=8)
        nlist = min(100, int(np.sqrt(features.shape[0])))
        valid_sizes.append(n_points)
        
        if run_algos["Exact_CPU"]:
            index_exact_cpu = faiss.IndexFlatL2(features.shape[1])
            index_exact_cpu.add(features)
            results_time["Exact_CPU"].append(measure_execution_time(lambda: index_exact_cpu.search(features, k_neighbors)))
            del index_exact_cpu
            current_task += 1
            print(f"[{current_task}/{total_tasks}] Exact_CPU zakończone")

        if run_algos["Exact_GPU"]:
            index_exact_gpu = faiss.IndexFlatL2(features.shape[1])
            index_exact_gpu = faiss.index_cpu_to_gpu(gpu_res, 0, index_exact_gpu)
            index_exact_gpu.add(features)
            results_time["Exact_GPU"].append(measure_execution_time(lambda: index_exact_gpu.search(features, k_neighbors)))
            del index_exact_gpu
            current_task += 1
            print(f"[{current_task}/{total_tasks}] Exact_GPU zakończone")

        if run_algos["IVFFlat"]:
            model_ivf = FaissIVFFlat(nlist=nlist, nprobe=params["IVFFlat"]["nprobe"], use_gpu=USE_GPU)
            model_ivf.build(features)
            results_time["IVFFlat"].append(measure_execution_time(lambda: model_ivf.query(features, k_neighbors)))
            del model_ivf
            current_task += 1
            print(f"[{current_task}/{total_tasks}] IVFFlat zakończone")
        
        if run_algos["IVFPQ"]:
            model_pq = FaissIVFPQ(nlist=nlist, m=params["IVFPQ"]["m"], nbits=8, nprobe=params["IVFPQ"]["nprobe"], use_gpu=USE_GPU)
            model_pq.build(features)
            results_time["IVFPQ"].append(measure_execution_time(lambda: model_pq.query(features, k_neighbors)))
            del model_pq
            current_task += 1
            print(f"[{current_task}/{total_tasks}] IVFPQ zakończone")

        if run_algos["HNSW"]:
            model_hnsw = HnswGraph(m=params["HNSW"]["m_graph"], ef_construction=200, ef=params["HNSW"]["ef"])
            model_hnsw.build(features)
            results_time["HNSW"].append(measure_execution_time(lambda: model_hnsw.query(features, k_neighbors)))
            del model_hnsw
            current_task += 1
            print(f"[{current_task}/{total_tasks}] HNSW zakończone")
        
        gc.collect()
        
    gpu_label = "GPU" if USE_GPU else "CPU"
    plot_ann_scaling(
        valid_sizes, 
        results_time, 
        use_gpu=USE_GPU,
        title="Przestrzeń 8D: Algorytmy Exact (100%) vs Przybliżone ANN"
    )

=== START EKSPERYMENTU: TIME vs N (USE_GPU dla FAISS ANN = True) ===

Ładowanie zbioru danych 1k...
[1/12] IVFFlat zakończone
[2/12] IVFPQ zakończone
[3/12] HNSW zakończone

Ładowanie zbioru danych 10k...
[4/12] IVFFlat zakończone


WARNING clustering 995 points to 31 centroids: please provide at least 1209 training points
WARNING clustering 995 points to 31 centroids: please provide at least 1209 training points
WARNING clustering 995 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 995 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 995 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 995 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 995 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 995 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 995 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 995 points to 256 centroids: please provide at least 9984 training points


[5/12] IVFPQ zakończone
[6/12] HNSW zakończone

Ładowanie zbioru danych 100k...
[7/12] IVFFlat zakończone
[8/12] IVFPQ zakończone
[9/12] HNSW zakończone

Ładowanie zbioru danych 1M...
[10/12] IVFFlat zakończone
[11/12] IVFPQ zakończone
[12/12] HNSW zakończone


<Figure size 1000x600 with 1 Axes>

# Weryfikacja Założeń: Exact kNN vs ANN w Niskich Wymiarach
Ostatni etap analizy to weryfikacja celowości stosowania algorytmów heurystycznych (ANN) w przestrzeniach o bardzo małej liczbie wymiarów (5D). Do zestawienia dołączono precyzyjne metody `FAISS ExactL2` oraz strukturę podziału przestrzeni `Scipy cKDTree` (100% Recall).

In [8]:
target_sizes = {"10k": 10000, "100k": 100000, "1M": 1000000}
mode = "hard"
k_neighbors = 5

run_algos = {
    "Exact_CPU": False,
    "Exact_GPU": False,
    "cKDTree_CPU": True,
    "IVFFlat_GPU": True,
    "HNSW_CPU": True
}

results_time = {alg: [] for alg in run_algos.keys()}
valid_sizes = []

print("=== START EKSPERYMENTU: EXACT kNN vs ANN (Przestrzeń 5D) ===")

valid_datasets_count = sum(1 for size in target_sizes if (data_dir / f"dataset_{mode}_{size}.npz").exists())
active_algos_count = sum(run_algos.values())
total_tasks = valid_datasets_count * active_algos_count
current_task = 0

if total_tasks == 0:
    print("[BŁĄD] Brak zadań do wykonania (brak plików lub wszystkie algorytmy wyłączone).")
else:
    if run_algos["Exact_GPU"] or run_algos["IVFFlat_GPU"]:
        gpu_res = faiss.StandardGpuResources()

    for size_label, n_points in target_sizes.items():
        dataset_path = data_dir / f"dataset_{mode}_{size_label}.npz"
        if not dataset_path.exists():
            print(f"[POMIJAM] Brak pliku: {size_label}")
            continue
            
        print(f"\nŁadowanie zbioru danych {size_label} (Surowe 5D)...")
        features = np.ascontiguousarray(np.load(dataset_path)["X"], dtype=np.float32)
        valid_sizes.append(n_points)
        
        nlist = min(100, int(np.sqrt(features.shape[0])))
        
        if run_algos["Exact_CPU"]:
            index_exact_cpu = faiss.IndexFlatL2(features.shape[1])
            index_exact_cpu.add(features)
            results_time["Exact_CPU"].append(measure_execution_time(lambda: index_exact_cpu.search(features, k_neighbors)))
            del index_exact_cpu
            current_task += 1
            print(f"[{current_task}/{total_tasks}] Exact_CPU zakończone")

        if run_algos["Exact_GPU"]:
            index_exact_gpu = faiss.IndexFlatL2(features.shape[1])
            index_exact_gpu = faiss.index_cpu_to_gpu(gpu_res, 0, index_exact_gpu)
            index_exact_gpu.add(features)
            results_time["Exact_GPU"].append(measure_execution_time(lambda: index_exact_gpu.search(features, k_neighbors)))
            del index_exact_gpu
            current_task += 1
            print(f"[{current_task}/{total_tasks}] Exact_GPU zakończone")
        
        if run_algos["cKDTree_CPU"]:
            results_time["cKDTree_CPU"].append(measure_execution_time(lambda: knn_scipy_ckdtree(features, k_neighbors)))
            current_task += 1
            print(f"[{current_task}/{total_tasks}] cKDTree_CPU zakończone")

        if run_algos["IVFFlat_GPU"]:
            model_ivf = FaissIVFFlat(nlist=nlist, nprobe=20, use_gpu=True)
            model_ivf.build(features)
            results_time["IVFFlat_GPU"].append(measure_execution_time(lambda: model_ivf.query(features, k_neighbors)))
            del model_ivf
            current_task += 1
            print(f"[{current_task}/{total_tasks}] IVFFlat_GPU zakończone")

        if run_algos["HNSW_CPU"]:
            model_hnsw = HnswGraph(m=16, ef_construction=200, ef=20)
            model_hnsw.build(features)
            results_time["HNSW_CPU"].append(measure_execution_time(lambda: model_hnsw.query(features, k_neighbors)))
            del model_hnsw
            current_task += 1
            print(f"[{current_task}/{total_tasks}] HNSW_CPU zakończone")

        gc.collect()

    plot_exact_vs_ann(valid_sizes, results_time, title="Wymiar 5D: Czy opłaca się używać metod przybliżonych (ANN)?")

=== START EKSPERYMENTU: EXACT kNN vs ANN (Przestrzeń 5D) ===

Ładowanie zbioru danych 10k (Surowe 5D)...
[1/9] cKDTree_CPU zakończone
[2/9] IVFFlat_GPU zakończone
[3/9] HNSW_CPU zakończone

Ładowanie zbioru danych 100k (Surowe 5D)...
[4/9] cKDTree_CPU zakończone
[5/9] IVFFlat_GPU zakończone
[6/9] HNSW_CPU zakończone

Ładowanie zbioru danych 1M (Surowe 5D)...
[7/9] cKDTree_CPU zakończone
[8/9] IVFFlat_GPU zakończone
[9/9] HNSW_CPU zakończone


<Figure size 1000x600 with 1 Axes>

## Wnioski Końcowe (Curse of Dimensionality)

Wyniki powyższego starcia dostarczają kluczowych wniosków inżynieryjnych:

1. **Deklasacja przez metody oparte na drzewach:** W przestrzeni 5-wymiarowej matematycznie dokładny algorytm `cKDTree` (złożoność $O(N \log N)$) deklasuje zaawansowane metody przybliżone.
2. **Metody ANN:** Z metod przybliżonych ANN IVFFlat zdaje się radzić sobie najlepiej. Możliwe, że dla zbiorów danych o większej liczebności lub liczbie wymiarów uzyskałby wyniki lepsze od cKDTree 


# Eksperyment: Czas w funkcji wymiarowości (D) i rozmiaru danych (N)

Poprzednie eksperymenty badały skalowanie względem N przy ustalonym D (5D lub 8D po
Zero-Paddingu). Tutaj obie zmienne zmieniają się jednocześnie: D ∈ {2, 4, 8} oraz
N ∈ {1k, 10k, 100k, 1M}.

Dla D=2 i D=4 wykorzystywane są pierwsze D kolumn oryginalnych 5-wymiarowych cech
(obcięcie — dopełnianie zerami "w dół" nie miałoby sensu fizycznego). Dla D=8
zachowana jest metodologia Zero-Padding z poprzednich sekcji. Dla IVFPQ liczba
subkwantyzatorów `m` jest zawsze równa D (musi dzielić D bez reszty), analogicznie
do sekcji "Granica Pareto", gdzie dla D=8 użyto m=8.

In [15]:
print("=== EKSPERYMENT: CZAS vs WYMIAROWOŚĆ (D) x ROZMIAR DANYCH (N) ===")

target_sizes = {"1k": 1000, "10k": 10000, "100k": 100000, "1M": 1000000}
dims_to_test = [2, 4, 8]
mode = "hard"
k_neighbors = 5

algo_factories = {
    "IVFFlat": lambda d, nlist: FaissIVFFlat(nlist=nlist, nprobe=20, use_gpu=USE_GPU),
    "IVFPQ":   lambda d, nlist: FaissIVFPQ(nlist=nlist, m=d, nprobe=20, use_gpu=USE_GPU),
    "HNSW":    lambda d, nlist: HnswGraph(m=16, ef_construction=200, ef=20),
}

# results_dim_scaling[algorytm][wymiarowość][N] = czas zapytania w sekundach
results_dim_scaling = {algo: {d: {} for d in dims_to_test} for algo in algo_factories}

for size_label, n_points in target_sizes.items():
    filename = data_dir / f"dataset_{mode}_{size_label}.npz"
    if not filename.exists():
        print(f"[POMINIĘTO] Plik {filename.name} nie istnieje.")
        continue

    print(f"\n--- N = {n_points:,} ---")
    features_5d = np.load(filename)["X"]
    nlist = min(100, int(np.sqrt(n_points)))  # jak w sekcji "Niższe wartości N"

    for d in dims_to_test:
        if d <= features_5d.shape[1]:
            features_d = np.ascontiguousarray(features_5d[:, :d], dtype=np.float32)
        else:
            features_d = pad_features(features_5d, target_dim=d)

        print(f"  D={d} (kształt: {features_d.shape})")

        for algo_name, make_model in algo_factories.items():
            model = make_model(d, nlist)

            measure_execution_time(lambda: model.build(features_d), num_runs=1, warmup_runs=0)
            query_time = measure_execution_time(lambda: model.query(features_d, k_neighbors), num_runs=3, warmup_runs=1)

            results_dim_scaling[algo_name][d][n_points] = query_time
            print(f"    {algo_name}: query={query_time:.4f}s")

        del features_d
        gc.collect()

=== EKSPERYMENT: CZAS vs WYMIAROWOŚĆ (D) x ROZMIAR DANYCH (N) ===

--- N = 1,000 ---
  D=2 (kształt: (995, 2))
    IVFFlat: query=0.0004s
    IVFPQ: query=0.0004s
    HNSW: query=0.0026s


WARNING clustering 995 points to 31 centroids: please provide at least 1209 training points
WARNING clustering 995 points to 31 centroids: please provide at least 1209 training points
WARNING clustering 995 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 995 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 995 points to 31 centroids: please provide at least 1209 training points
WARNING clustering 995 points to 31 centroids: please provide at least 1209 training points
WARNING clustering 995 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 995 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 995 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 995 points to 256 centroids: please provide at least 9984 training points


  D=4 (kształt: (995, 4))
    IVFFlat: query=0.0004s
    IVFPQ: query=0.0005s
    HNSW: query=0.0030s
  D=8 (kształt: (995, 8))
    IVFFlat: query=0.0004s


WARNING clustering 995 points to 31 centroids: please provide at least 1209 training points
WARNING clustering 995 points to 31 centroids: please provide at least 1209 training points
WARNING clustering 995 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 995 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 995 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 995 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 995 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 995 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 995 points to 256 centroids: please provide at least 9984 training points
WARNING clustering 995 points to 256 centroids: please provide at least 9984 training points


    IVFPQ: query=0.0007s
    HNSW: query=0.0032s

--- N = 10,000 ---
  D=2 (kształt: (9995, 2))
    IVFFlat: query=0.0031s
    IVFPQ: query=0.0031s
    HNSW: query=0.0286s
  D=4 (kształt: (9995, 4))
    IVFFlat: query=0.0034s
    IVFPQ: query=0.0044s
    HNSW: query=0.0360s
  D=8 (kształt: (9995, 8))
    IVFFlat: query=0.0032s
    IVFPQ: query=0.0065s
    HNSW: query=0.0400s

--- N = 100,000 ---
  D=2 (kształt: (99950, 2))
    IVFFlat: query=0.0670s
    IVFPQ: query=0.0559s
    HNSW: query=0.3620s
  D=4 (kształt: (99950, 4))
    IVFFlat: query=0.0767s
    IVFPQ: query=0.0760s
    HNSW: query=0.4323s
  D=8 (kształt: (99950, 8))
    IVFFlat: query=0.0850s
    IVFPQ: query=0.1040s
    HNSW: query=0.4549s

--- N = 1,000,000 ---
  D=2 (kształt: (999500, 2))
    IVFFlat: query=2.3257s
    IVFPQ: query=3.4136s
    HNSW: query=3.9778s
  D=4 (kształt: (999500, 4))
    IVFFlat: query=3.7897s
    IVFPQ: query=4.4023s
    HNSW: query=4.2586s
  D=8 (kształt: (999500, 8))
    IVFFlat: query=7.2597s


In [16]:
plot_time_dimension_heatmap(
    results_dim_scaling,
    dims=dims_to_test,
    sizes=list(target_sizes.values()),
    title="Czas zapytania: Wymiarowość (D) vs Rozmiar danych (N)"
)

NameError: name 'plot_time_dimension_heatmap' is not defined